# Spatial figures

1. Load and select one recording's pose and trials.
2. Plot body-position occupancy.
3. Define the circular arena using its centre and one boundary point in video pixels.
4. Check the arena overlay using `movement.roi.PolygonOfInterest.plot`.
5. Plot allocentric boundary bearing, egocentric boundary bearing, and radial head position.

In [10]:
##### Imports #########################################################
from pathlib import Path
import matplotlib.pyplot as plt
import movement.plots
import movement.kinematics
import movement.roi
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown, display



from data_conduit.datastructures import slice_stream, slice_stream_for_trial
from movement_figures.data_template.loading import build_datastructure

from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries
from movement_figures.video import read_session_video_frame



from data_conduit.refactor_qc import (                                          # Imports for trial filtering based on experiment protocol 
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    training_session_names,
    filter_trials,
)


from data_conduit.integrations.DLC.pose import pose_to_movement                 # Import for combining DLC pose & confidence streams into single object, matches `movement`'s conventions


### Notebook Settings ######################################################

pd.set_option("display.max_columns", None)                              # Changes default pandas display settings to show all columns in a dataframe.

In [11]:
##### Setup Parameters ################################################


# === 1| Select Mice, Days and Recording Folders Before Reading Files =============

ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                               # Example choices belong here: l0_selector and l1_selector.

# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                          # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                      # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                  # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                      # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering. 



In [12]:
##### Load DataStructure ###############################################################


#=== Load data ========================================================================
fig_ds = build_datastructure(
    root = ROOT,                                                                        # Directory above the named hierarchy levels.
    level_names = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams = ("trials", "session_settings", "video", "dlc"),
    include = None,                                                                     # Keep these session folder names; None keeps all.
    exclude = None,                                                                     # Alternatively omit these session folder names.
    raw_events = True,                                                                  # Convenience alias for adding "events" to the requested sources.
    device_yaml = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml = "./soundcard.yml",                                                 # Lab SoundCard register definitions.
    nosepoke_count = 18,                                                                # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                           # Preserve the existing lab parser setting.
    trial_spec = None,                                                                  # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.

    dlc_file_format = "auto",                                                           # Existing reader prefers CSV, otherwise HDF5.
    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds.select()
streams = fig_ds.load()



#=== Create convenient compact trial dataframe without some unnecessary columns. ======
#       More useful for visual inspection and analysis. 
#       Check the misc.py file for the default columns to keep and drop. 

streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                            # Whether to modify the trial dataframe in place or return a new dataframe. 
)


#=== Apply Trial Filtering Protocol to construct new stream objects ===================
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"], #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], #selectors={"session": session_id}
    ),
)
display(streams['dlc:movement'])


#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,
    show_stream_shapes = True,
    show_stream_dtypes = True,
    display_streams = False
)

     mouseID  training_day            session  n_total  min_trial  n_after_cut  exclude_on  n_on_dropped  n_kept
MR_M01569515             3 2026-06-21T093349Z      106         24           83        True             3      80
     mouseID  training_day            session  n_total  min_trial  n_after_cut  exclude_on  n_on_dropped  n_kept
MR_M01569515             3 2026-06-21T093349Z      106         24           83        True             3      80


<xarray.Dataset> Size: 8MB
Dimensions:     (individual: 1, keypoint: 5, space: 2, time: 36354)
Coordinates:
  * individual  (individual) object 8B 'individual_0'
  * keypoint    (keypoint) <U8 160B 'nose' 'lear' 'rear' 'body' 'tailbase'
  * space       (space) <U1 8B 'x' 'y'
  * time        (time) float64 291kB 5.633e+03 5.633e+03 ... 7.451e+03 7.451e+03
    session     (time) <U18 3MB '2026-06-21T093349Z' ... '2026-06-21T093349Z'
    mouseID     (time) object 291kB 'MR_M01569515' ... 'MR_M01569515'
    day         (time) object 291kB 'Day21' 'Day21' 'Day21' ... 'Day21' 'Day21'
Data variables:
    position    (time, space, keypoint, individual) float64 3MB 1.209e+03 ......
    confidence  (time, keypoint, individual) float64 1MB 0.9686 1.0 ... 0.9815
Attributes:
    source_software:  DeepLabCut
    time_unit:        seconds
    spatial_unit:     pixels

=========================
### Summary of Loaded Streams
=========================

,Stream Name,Shape,Type
0,session_settings:metadata,"(1, 23)",DataFrame
1,session_settings:trials,"(303, 132)",DataFrame
2,video,"(36354, 5)",DataFrame
3,dlc:position,"(36354, 5, 2)",DataArray
4,dlc:confidence,"(36354, 5)",DataArray
5,events,"(1487, 5)",DataFrame
6,trials,"(106, 28)",DataFrame
7,compact_trials,"(106, 19)",DataFrame
8,filtered_trials,"(80, 30)",DataFrame
9,filtered_compact_trials,"(80, 21)",DataFrame


In [13]:
from data_conduit.datastructures import slice_stream_per_trial, slice_stream


selected_trials = slice_stream(
    streams['filtered_trials'][:6]
    #streams["filtered_trials"], selectors={"trial_index": slice(1, 8)},
)


# Body positions for the recording.
point = streams['dlc:movement'].sel(keypoint="body", drop=True)

# Get positions for each complete selected trial.
selected = slice_stream_per_trial(
    stream=point,
    trials=selected_trials,
    segment="trial",
    time_coord="time",
)

# Get outbound positions using the trial table's outbound boundaries.
outbound = slice_stream_per_trial(
    stream=point,
    trials=selected_trials,
    segment="outbound",
    time_coord="time",
)

# Get inbound positions using the trial table's inbound boundaries.
inbound = slice_stream_per_trial(
    stream=point,
    trials=selected_trials,
    segment="inbound",
    time_coord="time",
)

# Each call above returns a list containing one position slice per trial.
# Join those slices along time to provide the positions for each heatmap.
occupancy_points = {
    "Selected trials": xr.concat(selected, dim="time"),
    "Outbound": xr.concat(outbound, dim="time"),
    "Inbound": xr.concat(inbound, dim="time"),
}

display(selected_trials)

# Select positions for every chosen trial, keeping all keypoints.
angle_positions = slice_stream_per_trial(
    stream=streams['dlc:movement'],
    trials=selected_trials,
    segment="trial",
    time_coord="time",
)

# Join the selected trial slices along time.
angle_positions = xr.concat(angle_positions, dim="time")



##### Video Background and Static ROI #####

# Read the video for the single recording represented by the selected trials.
session_id = selected_trials["session"].unique().item()
background_frame, video_path = read_session_video_frame(fig_ds.sessions[session_id].path)

,trial_index,start_time,end_time,start_inclusive,end_inclusive,first_event_index,last_event_index,tz_triggered_time,tz_available_time,ChosenPort,CorrectPort,outcome,LED,angle_offset,target_zone_size,TTT,TTP,outbound_start_time,outbound_end_time,outbound_start_inclusive,outbound_end_inclusive,inbound_start_time,inbound_end_time,inbound_start_inclusive,inbound_end_inclusive,session,mouseID,day,training_day,day_trial_index
0,24,5922.650944,5930.218816,False,True,332,345,5928.000992,5925.144000,0,0,Success,OFF,0.0,300.0,5.350048,2.217824,5922.650944,5928.000992,False,True,5928.000992,5930.218816,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,1
1,25,5930.218816,5937.076640,False,True,346,359,5935.200000,5932.724000,0,0,Success,OFF,0.0,300.0,4.981184,1.876640,5930.218816,5935.200000,False,True,5935.200000,5937.076640,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,2
2,26,5937.076640,5947.709248,False,True,360,373,5944.049984,5939.589984,0,0,Success,OFF,0.0,300.0,6.973344,3.659264,5937.076640,5944.049984,False,True,5944.049984,5947.709248,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,3
3,27,5947.709248,5956.329760,False,True,374,387,5953.200000,5950.220992,0,0,Success,OFF,0.0,300.0,5.490752,3.129760,5947.709248,5953.200000,False,True,5953.200000,5956.329760,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,4
4,28,5956.329760,5963.959072,False,True,388,401,5962.250976,5958.838976,0,0,Success,OFF,0.0,300.0,5.921216,1.708096,5956.329760,5962.250976,False,True,5962.250976,5963.959072,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,5
5,29,5963.959072,5980.614496,False,True,402,415,5975.602976,5970.956992,-1,0,Miss,OFF,NaN,150.0,11.643904,5.011520,5963.959072,5975.602976,False,True,5975.602976,5980.614496,True,True,2026-06-21T093349Z,MR_M01569515,Day21,3,6



---------------------------
# Figure Functions
----------------------------

In [14]:
##### Spatial Occupancy Plot #####

def plot_spatial_occupancy(
    points, 
    background_frame, 
  #  fixed_roi_px, 
    *, 
    # roi_label="North / top of video",
    bins=50, 
    vmax=100.0,
    cmap = "magma",
    cmap_norm = "linear", 
    alpha=0.70,
):
    """Plot body-position occupancy over the first video frame using movement.

    Each panel counts the selected position samples falling inside each spatial
    bin. This is a sample-count map: it is not normalised to seconds or to a
    probability. Trial selection and any confidence filtering happen earlier.

    Parameters
    ----------
    points : dict[str, xarray.Dataset]
        Panel labels mapped to selected pose datasets, as in occupancy_points.
        Each dataset contains position in video pixels for the body keypoint.
        The arrays and background must belong to the same recording. movement
        omits positions with missing coordinates; binning covers the video area.
    background_frame : numpy.ndarray
        First RGB frame of the matching video, with shape (height, width, 3).
        Its dimensions set the common image extent and histogram boundaries.
    fixed_roi_px : array-like
        ROI marker (x, y) in video pixels, shared with the static-bearing figure.
        The marker identifies the target; it does not change the sample counts.
    roi_label : str
        Target name displayed in each panel's legend.
    bins : int
        Number of equal-width bins along each image axis. For example, 30
        produces a 30 by 30 grid covering the full video frame.
    vmax : float
        Upper colour limit in samples per bin, used for every panel. Larger
        counts use the top colour but retain their full values in occupancy.
    alpha : float
        Heatmap opacity from 0 (transparent) to 1 (opaque). movement supplies
        each panel's colour bar and normal rendering, including zero-count bins.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Occupancy figure with one colour bar per panel, ready to display or save.
    axes : numpy.ndarray
        One-dimensional array of panel axes, in the order supplied by points.
    occupancy : dict
        movement's histogram result for each panel label: h holds raw counts
        indexed by x bin then y bin; xedges and yedges hold the bin boundaries
        in pixels. Each edge array has one more value than the number of bins.
    """
    # --- Match the panels and histogram bins to the video ---
    height, width = background_frame.shape[:2]
    extent = (-0.5, width - 0.5, height - 0.5, -0.5)
    bin_range = ((-0.5, width - 0.5), (-0.5, height - 0.5))
    fig, grid = plt.subplots(
        1, len(points), figsize=(16, 6), squeeze=False,
        sharex=True, sharey=True, layout="constrained",
    )
    axes = grid[0]
    occupancy = {}

    # --- Plot occupancy and the ROI marker ---
    for ax, (label, point) in zip(axes, points.items()):
        ax.imshow(background_frame, origin="upper", extent=extent)
        _, _, occupancy[label] = movement.plots.plot_occupancy(
            da=point["position"],                                   # Coordinates from the selected pose dataset.
            ax=ax, 
            bins=bins, 
            range=bin_range,
            cmap=cmap, 
            vmin=0, 
            vmax=vmax, 
            alpha=alpha,
            norm = cmap_norm
        )
        # ax.scatter(
        #     *fixed_roi_px, marker="*", s=110, color="gold",
        #     edgecolor="black", label=roi_label,
        # )
        # ax.legend()
        ax.set(
            title=label, xlabel="x (px)", ylabel="y (px)", aspect="equal",
            xlim=extent[:2], ylim=extent[2:],  # Video y increases downwards.
        )

    fig.suptitle("Body-position occupancy (samples per bin)")
    return fig, axes, occupancy


In [15]:
##### Return the Occupancy Figure #####

occupancy_figure, occupancy_axes, occupancy = plot_spatial_occupancy(
    points=occupancy_points,
    background_frame=background_frame,
)

occupancy_figure.scatter(x= 

)
plt.close(occupancy_figure)  # The last line displays the returned figure once.
occupancy_figure


SyntaxError: expected argument value expression (3580794835.py, line 8)

In [ ]:
##### Arena Geometry and Shared Plot Helpers #####

def make_circular_arena_roi(centre_px, boundary_point_px, *, n_vertices=720):
    """Return (movement polygon ROI, radius) from two measured (x, y) points.

    Both points must use the same image coordinates as the pose. The radius
    is the distance from centre_px to boundary_point_px. No arena geometry
    is inferred from the video dimensions. The polygon approximates a circle;
    with 720 vertices, its maximum wall-distance error is below 0.00001 * R.
    """
    centre = np.asarray(centre_px, dtype=float)
    boundary = np.asarray(boundary_point_px, dtype=float)
    if centre.shape != (2,) or boundary.shape != (2,) or not np.isfinite([centre, boundary]).all():
        raise ValueError("Set arena_centre_px and arena_boundary_point_px to finite (x, y) coordinates.")
    radius = float(np.linalg.norm(boundary - centre))
    if radius <= 0:
        raise ValueError("The boundary point must differ from the arena centre.")
    if not isinstance(n_vertices, (int, np.integer)) or n_vertices < 16:
        raise ValueError("n_vertices must be an integer of at least 16.")
    theta = np.linspace(0, 2 * np.pi, n_vertices, endpoint=False)
    vertices = centre + radius * np.column_stack((np.cos(theta), np.sin(theta)))
    return movement.roi.PolygonOfInterest(vertices, name="Arena"), radius


def _circular_head_geometry(angle_positions, individual, arena_centre_px, arena_radius_px, left_ear, right_ear):
    """Select the ear midpoint and identify valid positions in the circle."""
    centre = np.asarray(arena_centre_px, dtype=float)
    if centre.shape != (2,) or not np.isfinite(centre).all():
        raise ValueError("arena_centre_px must contain finite (x, y) coordinates.")
    if not np.isscalar(arena_radius_px) or not np.isfinite(arena_radius_px) or arena_radius_px <= 0:
        raise ValueError("arena_radius_px must be positive and finite.")
    position = angle_positions["position"].sel(individual=individual, drop=True).sel(space=["x", "y"])
    head_centre = (
        position.sel(keypoint=left_ear, drop=True)
        + position.sel(keypoint=right_ear, drop=True)
    ).transpose("time", "space") / 2
    time = np.asarray(head_centre.time, dtype=float)
    if not time.size or not np.isfinite(time).all() or (np.diff(time) < 0).any():
        raise ValueError("Pose time must contain finite, nondecreasing recording seconds.")
    if "session" in head_centre.coords and pd.unique(head_centre.session.values.ravel()).size > 1:
        raise ValueError("Select pose from one recording before defining its arena ROI.")
    radial = np.hypot(
        head_centre.sel(space="x", drop=True) - centre[0],
        head_centre.sel(space="y", drop=True) - centre[1],
    )
    tolerance = 1e-10 * arena_radius_px
    inside = np.isfinite(head_centre).all("space") & (radial <= arena_radius_px + tolerance)
    # At the circle centre there is no unique nearest wall direction. At the
    # wall, the approach vector has zero length. Neither defines a bearing.
    angle_valid = inside & (radial > tolerance) & (radial < arena_radius_px - tolerance)
    return position, head_centre, radial, inside, angle_valid



def _interior_bearing_indices(arena_roi, head_centre, valid):
    """Exclude the thin gap between the ideal circle and its polygon ROI.

    A point outside the polygon sees its nearest boundary inward, which would
    reverse the intended circular-arena bearing. Edge points are also undefined.
    """
    indices = np.flatnonzero(valid.values)
    if indices.size:
        contained = arena_roi.contains_point(head_centre.isel(time=indices), include_boundary=False)
        indices = indices[np.asarray(contained, dtype=bool)]
    return indices


def _empty_head_trace(head_centre, name, units):
    """Keep every input timestamp, including repeated trial-boundary samples."""
    result = xr.full_like(head_centre.sel(space="x", drop=True), np.nan, dtype=float).rename(name)
    result.attrs = {"units": units, "head_position": "midpoint of left and right ears"}
    return result


def _plot_spatial_trace(trace, selected_trials, *, row_duration_s, ylabel, title, ylim, angular=False):
    """Draw annotated, equal-duration rows without joining gaps or angle wraps."""
    if not np.isscalar(row_duration_s) or not np.isfinite(row_duration_s) or row_duration_s <= 0:
        raise ValueError("row_duration_s must be positive and finite.")
    if "session" in trace.coords and "session" in selected_trials:
        trial_sessions = selected_trials["session"].dropna().unique()
        pose_sessions = pd.unique(trace.session.values.ravel())
        if len(trial_sessions) and (len(trial_sessions) != 1 or trial_sessions[0] != pose_sessions[0]):
            raise ValueError("Pose and selected_trials must belong to the same recording.")
    start, end = float(trace.time[0]), float(trace.time[-1])
    row_count = max(1, int(np.ceil((end - start) / row_duration_s - 1e-12)))
    fig, grid = plt.subplots(
        row_count, 1, figsize=(13, 2.6 * row_count),
        squeeze=False, sharey=True, layout="constrained",
    )
    axes = grid[:, 0]
    for row, ax in enumerate(axes):
        row_start = start + row * row_duration_s
        row_end = row_start + row_duration_s
        ax.plot(trace.time, trace, ".", ms=2.5, color="#5b4b9a")
        ax.set(xlim=(row_start, row_end), ylim=ylim, xlabel="Recording time (s)", ylabel=ylabel)
        if angular:
            ax.set_yticks([-np.pi, 0, np.pi], ["−π", "0", "π"])
        ax.grid(axis="y", alpha=0.2)
        annotate_qc_timeseries(ax, selected_trials, window=(row_start, row_end))
    fig.suptitle(title)
    return fig, axes, trace


##### Allocentric Head Position: Bearing to the Arena Boundary #####

def plot_allocentric_head_direction(
    angle_positions, selected_trials, individual, arena_roi, arena_centre_px, arena_radius_px, *,
    left_ear="lear", right_ear="rear", reference_vector=(1.0, 0.0), row_duration_s=25.0,
):
    """Plot the allocentric bearing from the ear midpoint to the nearest wall.

    Pass the ROI/radius returned by make_circular_arena_roi and its centre.
    This is the wall's direction from the head position, independent of which
    way the head faces. movement measures the signed angle from the approach
    vector TO reference_vector. With image y down and reference (1, 0), right
    is 0, top is +pi/2, bottom is -pi/2, and left is +/-pi.

    Input pose/trials must already be selected to one recording; all positions
    and arena geometry use video pixels. Returns (figure, row axes, angles in
    radians), preserving timestamps. Missing positions, positions outside the
    circle, and its centre/wall have NaN bearings. Coincident ears can still
    define a position, although they cannot define egocentric head direction.
    """
    _, head_centre, _, _, valid = _circular_head_geometry(
        angle_positions, individual, arena_centre_px, arena_radius_px, left_ear, right_ear,
    )
    reference = np.asarray(reference_vector, dtype=float)
    if reference.shape != (2,) or not np.isfinite(reference).all() or not np.any(reference):
        raise ValueError("reference_vector must be a finite, nonzero (x, y) vector.")
    head_angles = _empty_head_trace(head_centre, "allocentric_boundary_angle", "rad")
    indices = _interior_bearing_indices(arena_roi, head_centre, valid)
    if indices.size:
        # Filter before calling ROI: Shapely cannot calculate a nearest point
        # for NaN/inf inputs. Restore results by sample index (times can repeat).
        measured = arena_roi.compute_allocentric_angle_to_nearest_point(
            head_centre.isel(time=indices), boundary_only=True,
            reference_vector=reference, in_degrees=False,
        )
        head_angles.values[indices] = measured.values
    return _plot_spatial_trace(
        head_angles, selected_trials, row_duration_s=row_duration_s,
        ylabel="Allocentric bearing (rad)", title="Allocentric bearing to arena boundary",
        ylim=(-np.pi, np.pi), angular=True,
    )

In [ ]:
##### Egocentric Head Direction: Static Arena ROI #####

def plot_egocentric_static_head_direction(
    angle_positions, selected_trials, individual, arena_roi, arena_centre_px, arena_radius_px, *,
    left_ear="lear", right_ear="rear", camera_view="top_down", row_duration_s=25.0,
):
    """Plot the nearest wall's bearing relative to the head-forward direction.

    Use the same circular ROI, centre and radius as the allocentric plot.
    The arena is static; its nearest point changes as the animal moves.
    movement supplies both the head-forward vector and the ROI bearing.
    With top-down video coordinates, zero means facing the nearest wall,
    positive means wall to the animal's left, and negative means wall to its
    right (these sides reverse for bottom_up). The angle is measured from the approach vector TO head-forward.

    Pose/trials are preselected to one recording. left_ear/right_ear are
    anatomical keypoints; camera_view is movement's top_down or bottom_up.
    Returns (figure, row axes, angles in radians). Invalid positions, points
    outside the circle, circle centre/wall and coincident ears return NaN.
    Every supplied timestamp is retained, including repeated trial endpoints.
    """
    if camera_view not in {"top_down", "bottom_up"}:
        raise ValueError("camera_view must be top_down or bottom_up.")
    position, head_centre, _, _, valid = _circular_head_geometry(
        angle_positions, individual, arena_centre_px, arena_radius_px, left_ear, right_ear,
    )
    head_forward = movement.kinematics.compute_head_direction_vector(
        data=position.where(np.isfinite(position)), left_keypoint=left_ear, right_keypoint=right_ear,
        camera_view=camera_view,
    )
    valid = valid & np.isfinite(head_forward).all("space") & (np.abs(head_forward).sum("space") > 0)
    head_angles = _empty_head_trace(head_centre, "egocentric_boundary_angle", "rad")
    indices = _interior_bearing_indices(arena_roi, head_centre, valid)
    if indices.size:
        measured = arena_roi.compute_egocentric_angle_to_nearest_point(
            direction=head_forward.isel(time=indices),
            position=head_centre.isel(time=indices),
            boundary_only=True, in_degrees=False,
        )
        head_angles.values[indices] = measured.values
    return _plot_spatial_trace(
        head_angles, selected_trials, row_duration_s=row_duration_s,
        ylabel="Egocentric bearing (rad)", title="Egocentric bearing to arena boundary",
        ylim=(-np.pi, np.pi), angular=True,
    )

## Define and check the arena

Set `arena_centre_px = (centre_x, centre_y)` and
`arena_boundary_point_px = (wall_x, wall_y)` in the next cell using the matching
video/pose coordinate system. Their separation gives the radius. If you know
the radius directly, use `(centre_x + radius_px, centre_y)` for the boundary point.
The `None` placeholders deliberately require your geometry before these plots run.

All three plots use the **same static circular arena**, with head position at the
midpoint of the ears. `individual="individual_0"` selects the animal in the loaded
pose dataset. Time is recording seconds; each row shows 25 seconds.

| Figure | Calculation | Meaning |
| --- | --- | --- |
| Allocentric | `arena_roi.compute_allocentric_angle_to_nearest_point(...)` | Bearing from head position to nearest wall, relative to image-right; positive towards image-top |
| Egocentric | `arena_roi.compute_egocentric_angle_to_nearest_point(...)` | Nearest wall relative to head-forward; 0 facing the wall, positive left in top-down video |
| Distance | Centre-to-head Euclidean distance; also `arena_roi.compute_distance_to(...)` | Plotted radius is 0 at centre and R at boundary; returned `distance_to_boundary` is 0 at the wall |

Each ROI call uses `boundary_only=True`: a filled polygon otherwise has zero
distance for every interior point. A 720-sided polygon approximates the circle.
For an ideal circle, radial position = R − wall distance; the plotted radial
position is calculated directly so that polygon approximation does not shift
zero at the centre. Angles at the centre/wall are undefined; missing and
outside-arena positions remain NaN. Angular samples in the tiny gap between
the circle and its inscribed polygon are masked to avoid reversed bearings.
Angles use movement's native ROI sign,
which is the reverse of the old manual head-to-target signed-angle call.

`arena_roi.plot(ax=...)` draws the geometry for checking its alignment with the
video; it does not create the angle/distance timeseries.
See the [movement ROI example](https://movement.neuroinformatics.dev/latest/examples/boundary_angles.html).

In [ ]:
##### Set Arena Coordinates and Check the ROI Overlay #####

# Replace these with measured (x, y) coordinates in the video/pose pixels.
arena_centre_px = None         # (centre_x, centre_y)
arena_boundary_point_px = None # (wall_x, wall_y), any one point on the circular wall

arena_roi, arena_radius_px = make_circular_arena_roi(
    centre_px=arena_centre_px, boundary_point_px=arena_boundary_point_px,
)

# background_frame was read from the selected recording above.
arena_figure, arena_ax = plt.subplots(figsize=(7, 7), layout="constrained")
arena_ax.imshow(background_frame, origin="upper")
arena_roi.plot(ax=arena_ax, facecolor="none", edgecolor="cyan", linewidth=2)
arena_ax.scatter(*arena_centre_px, marker="+", color="cyan", label="Arena centre")
arena_ax.scatter(*arena_boundary_point_px, marker="x", color="gold", label="Measured wall point")
arena_ax.set(
    title=f"Arena ROI: radius = {arena_radius_px:.2f} px", aspect="equal",
    xlabel="x (px)", ylabel="y (px)",
    xlim=(-0.5, background_frame.shape[1] - 0.5),
    ylim=(background_frame.shape[0] - 0.5, -0.5),
)
arena_ax.legend()
plt.close(arena_figure)
arena_figure

ValueError: Set arena_centre_px and arena_boundary_point_px to finite (x, y) coordinates.

In [ ]:
##### Allocentric Boundary Distance: Radial Head Position #####


def plot_allocentric_boundary_distance(
    angle_positions, selected_trials, individual, arena_roi, arena_centre_px, arena_radius_px, *,
    left_ear="lear", right_ear="rear", row_duration_s=25.0,
):
    """Plot head distance from the arena centre: 0 at centre, radius at wall.

    Use the ROI/radius returned by make_circular_arena_roi and its centre,
    all in pose pixels. Returns (figure, row axes, radial distance DataArray).
    The trace also carries a distance_to_boundary coordinate calculated with
    arena_roi.compute_distance_to(..., boundary_only=True), in pixels.

    Distance to the wall decreases as radial position increases. For an ideal
    circle r = R - distance_to_wall. We calculate r directly from the centre
    to avoid the small polygon approximation error in the ROI's wall distance.
    Missing and outside-arena positions remain NaN, without clipping them to
    the plot range. Coincident ears still define a usable midpoint/distance.
    """
    _, head_centre, radial, valid, _ = _circular_head_geometry(
        angle_positions, individual, arena_centre_px, arena_radius_px, left_ear, right_ear,
    )
    boundary_distance = _empty_head_trace(head_centre, "distance_to_boundary", "px")
    indices = np.flatnonzero(valid.values)
    if indices.size:
        measured = arena_roi.compute_distance_to(
            head_centre.isel(time=indices), boundary_only=True,
        )
        boundary_distance.values[indices] = measured.values
    head_distance = _empty_head_trace(head_centre, "radial_head_distance", "px")
    head_distance.values[indices] = radial.values[indices]
    head_distance = head_distance.assign_coords(distance_to_boundary=boundary_distance)
    return _plot_spatial_trace(
        head_distance, selected_trials, row_duration_s=row_duration_s,
        ylabel="Distance from centre (px)", title="Head position: distance from arena centre",
        ylim=(0, arena_radius_px),
    )

In [ ]:
##### Return the Allocentric Boundary-Bearing Figure #####

allocentric_figure, allocentric_axes, allocentric_angles = plot_allocentric_head_direction(
    angle_positions=angle_positions,
    selected_trials=selected_trials,
    individual="individual_0",
    arena_roi=arena_roi,
    arena_centre_px=arena_centre_px,
    arena_radius_px=arena_radius_px,
    reference_vector=(1.0, 0.0),  # Right = 0; top = +pi/2 in image coordinates.
    row_duration_s=25.0,
)
plt.close(allocentric_figure)
allocentric_figure

In [ ]:
##### Return the Static Egocentric Boundary-Bearing Figure #####

static_figure, static_axes, static_angles = plot_egocentric_static_head_direction(
    angle_positions=angle_positions,
    selected_trials=selected_trials,
    individual="individual_0",
    arena_roi=arena_roi,
    arena_centre_px=arena_centre_px,
    arena_radius_px=arena_radius_px,
    camera_view="top_down",  # With image y down, positive bearing means wall to the left.
    row_duration_s=25.0,
)
plt.close(static_figure)
static_figure

In [ ]:
##### Return the Radial Head-Distance Figure #####

distance_figure, distance_axes, head_distance = plot_allocentric_boundary_distance(
    angle_positions=angle_positions,
    selected_trials=selected_trials,
    individual="individual_0",
    arena_roi=arena_roi,
    arena_centre_px=arena_centre_px,
    arena_radius_px=arena_radius_px,
    row_duration_s=25.0,
)
# head_distance is 0 at centre, R at wall.
# head_distance.coords["distance_to_boundary"] holds movement's wall distances.
plt.close(distance_figure)
distance_figure